In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from io import BytesIO
from azure.storage.blob import BlobServiceClient
from tqdm import tqdm

# Configuration
STORAGE_ACCOUNT = "solarflarestorageproject"
STORAGE_KEY = os.environ.get("AZURE_STORAGE_KEY", "")
RAW_CONTAINER = "raw"

print("Configuration loaded successfully.")

Configuration loaded successfully.


In [2]:
# Run this cell manually each session - do not save with key filled in
# Before running, paste the key below temporarily
import os
os.environ["AZURE_STORAGE_KEY"] = ""
STORAGE_KEY = os.environ.get("AZURE_STORAGE_KEY", "")

Key set for this session.


In [3]:
# Connect to Azure Blob Storage
container_client = BlobServiceClient(
    account_url=f"https://{STORAGE_ACCOUNT}.blob.core.windows.net",
    credential=STORAGE_KEY
).get_container_client(RAW_CONTAINER)

# List all blobs
blobs = list(container_client.list_blobs())
df = pd.DataFrame({"blob_name": [b.name for b in blobs]})

print(f"Connected successfully.")
print(f"Total images found: {len(df)}")

Connected successfully.
Total images found: 0


In [4]:
# Label parsing functions
def parse_label(blob_name):
    """Extract flare class from filename (X, M, C, B, or N for no flare)."""
    filename = blob_name.split("/")[-1]
    for cls in ["X", "M", "C", "B"]:
        if filename.startswith(cls):
            return cls
    return "N"

def parse_binary_label(label):
    """Binary classification: 1 = flare (M or X), 0 = no flare (B, C, N)."""
    return 1 if label in ["M", "X"] else 0

# Apply labels if data is available
if len(df) > 0:
    df["label"] = df["blob_name"].apply(parse_label)
    df["binary_label"] = df["label"].apply(parse_binary_label)
    print("Multi-class distribution:")
    print(df["label"].value_counts())
    print("\nBinary distribution:")
    print(df["binary_label"].value_counts())
    print(f"\nFlare rate: {df['binary_label'].mean():.2%}")
else:
    print("No data available yet. Cell will execute fully once dataset is uploaded.")

No data available yet. Cell will execute fully once dataset is uploaded.


In [5]:
def sample_pixel_stats(container_client, df, n_samples=100):
    """Sample n images and compute pixel-level statistics."""
    if len(df) == 0:
        return None
    
    sample = df.sample(min(n_samples, len(df)))
    stats = []

    for _, row in tqdm(sample.iterrows(), total=len(sample)):
        try:
            data = container_client.download_blob(row["blob_name"]).readall()
            img = Image.open(BytesIO(data)).convert("L")
            arr = np.array(img).astype(np.float32)
            stats.append({
                "blob_name":  row["blob_name"],
                "label":      row["label"],
                "mean_pixel": arr.mean(),
                "std_pixel":  arr.std(),
                "min_pixel":  arr.min(),
                "max_pixel":  arr.max(),
            })
        except Exception as e:
            print(f"Error processing {row['blob_name']}: {e}")
            continue

    return pd.DataFrame(stats)

if len(df) > 0:
    stats_df = sample_pixel_stats(container_client, df)
    print(stats_df.describe())
else:
    print("No data available yet. Cell will execute fully once dataset is uploaded.")

No data available yet. Cell will execute fully once dataset is uploaded.


In [6]:
def plot_class_distribution(df):
    """Plot multi-class and binary class distributions."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle("Solar Flare Dataset - Class Distribution", fontsize=14)

    # Multi-class
    df["label"].value_counts().plot(kind="bar", ax=axes[0], color="steelblue", edgecolor="black")
    axes[0].set_title("Multi-class Distribution")
    axes[0].set_xlabel("Flare Class")
    axes[0].set_ylabel("Count")
    axes[0].tick_params(axis="x", rotation=0)

    # Binary
    binary_counts = df["binary_label"].value_counts()
    binary_counts.index = ["No Flare (0)", "Flare (1)"] if 0 in binary_counts.index else ["Flare (1)", "No Flare (0)"]
    binary_counts.plot(kind="bar", ax=axes[1], color=["steelblue", "salmon"], edgecolor="black")
    axes[1].set_title("Binary Distribution")
    axes[1].set_xlabel("Class")
    axes[1].set_ylabel("Count")
    axes[1].tick_params(axis="x", rotation=0)

    plt.tight_layout()
    os.makedirs("outputs", exist_ok=True)
    plt.savefig("outputs/class_distribution.png", dpi=150)
    plt.show()
    print("Plot saved to outputs/class_distribution.png")

def plot_pixel_stats(stats_df):
    """Plot pixel intensity statistics by class."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle("Solar Flare Dataset - Pixel Statistics by Class", fontsize=14)

    sns.boxplot(data=stats_df, x="label", y="mean_pixel", ax=axes[0], palette="Blues")
    axes[0].set_title("Mean Pixel Intensity by Class")
    axes[0].set_xlabel("Flare Class")
    axes[0].set_ylabel("Mean Pixel Value")

    sns.boxplot(data=stats_df, x="label", y="std_pixel", ax=axes[1], palette="Blues")
    axes[1].set_title("Pixel Standard Deviation by Class")
    axes[1].set_xlabel("Flare Class")
    axes[1].set_ylabel("Standard Deviation")

    plt.tight_layout()
    plt.savefig("outputs/pixel_statistics.png", dpi=150)
    plt.show()
    print("Plot saved to outputs/pixel_statistics.png")

if len(df) > 0:
    plot_class_distribution(df)
    plot_pixel_stats(stats_df)
else:
    print("No data available yet. Cell will execute fully once dataset is uploaded.")

No data available yet. Cell will execute fully once dataset is uploaded.


In [7]:
def show_sample_images(container_client, df, n=8):
    """Display a grid of sample magnetogram images with their labels."""
    if len(df) == 0:
        print("No data available yet. Cell will execute fully once dataset is uploaded.")
        return

    sample = df.sample(min(n, len(df)))
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    fig.suptitle("Sample Solar Magnetogram Images by Flare Class", fontsize=14)
    axes = axes.flatten()

    for i, (_, row) in enumerate(sample.iterrows()):
        try:
            data = container_client.download_blob(row["blob_name"]).readall()
            img = Image.open(BytesIO(data)).convert("L")
            axes[i].imshow(img, cmap="gray")
            axes[i].set_title(f"Class: {row['label']}")
            axes[i].axis("off")
        except Exception as e:
            axes[i].set_title("Error loading image")
            axes[i].axis("off")

    plt.tight_layout()
    os.makedirs("outputs", exist_ok=True)
    plt.savefig("outputs/sample_images.png", dpi=150)
    plt.show()
    print("Sample images saved to outputs/sample_images.png")

show_sample_images(container_client, df)

No data available yet. Cell will execute fully once dataset is uploaded.
